<a href="https://colab.research.google.com/drive/1AqljuzQtSI97-juTuDTI9Uy3JGIrEqmH?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Investigating structures with data fetched from APIs

<img src="https://raw.githubusercontent.com/3D-Beacons/3D-Beacons/main/assets/3D-Beacons-logo.png" height="100" align="right">

## Introduction

Welcome to this Google Colab notebook, an essential companion to our paper on accessing data through the 3D-Beacons API.

This notebook serves as a practical resource for researchers, developers, and data enthusiasts who are interested in exploring the potential of the 3D-Beacons API and leveraging its capabilities. By following this guide, you will gain a deep understanding of how to interact with the 3D-Beacons API to access and analyze various types of data.
To supplement your learning, we have provided links to the full paper as well as documentation resources that will assist you in navigating the API effectively.

Documentation Link: [3D-Beacons API Documentation](https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/#/default/get_uniprot_summary_uniprot_summary__qualifier__json_get)

<br>

**Further readings:**

**3D-Beacons: Decreasing the gap between protein sequences and structures through a federated network of protein structure data resources**

Mihaly Varadi, Sreenath Nair, Ian Sillitoe, Gerardo Tauriello, Stephen Anyango, Stefan Bienert, Clemente Borges, Mandar Deshpande, Tim Green, Demis Hassabis, Andras Hatos, Tamas Hegedus, Maarten L Hekkelman, Robbie Joosten, John Jumper, Agata Laydon, Dmitry Molodenskiy, Damiano Piovesan, Edoardo Salladini, Steven L. Salzberg, Markus J Sommer, Martin Steinegger, Erzsebet Suhajda, Dmitri Svergun, Luiggi Tenorio-Ku, Silvio Tosatto, Kathryn Tunyasuvunakool, Andrew Mark Waterhouse, Augustin Žídek, Torsten Schwede, Christine Orengo, Sameer Velankar
3 August 2022; BioRxiv https://doi.org/10.1101/2022.08.01.501973


  ## Instructions <a name="INSTRUCTIONS"></a>

* Quick Start <a name="Quick Start"></a>

In order to make the learning experience more accessible and interactive, we have incorporated widgets that allow you to provide inputs and customize certain aspects of the code. These widgets serve as interactive elements that enhance your ability to interact with the code and observe the impact of different inputs on the results. Throughout this tutorial, you will come across code chunks that generate these interactive widgets.

  * How to use Google Colab <a name="Google Colab"></a>
1. Before running the code, make sure you have filled the necessary information in the input form.
2. To run a code cell, click on the cell to select it. You will notice a play button (▶️) on the left side of the cell. Click on the play button or press Shift+Enter to run the code in the selected cell.
3. The code will start executing, and you will see the output, if any, displayed below the code cell.
4. Move to the next code cell and repeat steps 2 and 3 until you have executed all the desired code cells in sequence.
5. The currently running step is indicated by a circle with a stop sign next to it.
If you need to stop or interrupt the execution of a code cell, you can click on the stop button (■) located next to the play button.


*Remember to run the code cells in the correct order, as their execution might depend on variables or functions defined in previous cells. You can modify the code in a code cell and re-run it to see updated results.*

*Note: If the notebook runtime is restarted you will need to re-run the first 3 code chunks located in the Set Up section.*






---



In [ ]:
#@title Set Up: Install libraries and mount Google Drive

#@markdown After running this code, you may be prompted to grant access to your Google Drive. This is necessary for Google Colab to download the files and save them to your Drive.
#@markdown <br>
#@markdown <br>
#@markdown Please follow the on-screen instructions to provide the necessary permissions, as it enables seamless integration between Colab and your Drive. Rest assured that your data and files are secure and will not be accessed without your explicit permission.

!pip install ijson gwpy &> /dev/null

print("Succesfully installed!")

#@title Mount Drive to download files
######## LIBRARIES
import ijson
import requests, sys, json
import ipywidgets as wgt
from urllib.request import urlopen
# Importing the necessary libraries for mounting Google Drive
import os
from google.colab import drive


######## FUNCTIONS
#defining functions for search and download
def Search3DBeacons(ID):
  WEBSITE_API = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/v2/uniprot/summary/"

  r = ijson.parse(urlopen(f"{WEBSITE_API}{ID}.json"))
  structures = list(ijson.items(r, "structures.item", use_float=True))
  return structures


# Defining function for sequence search
def sequence_search(sequence):
    # global job_id

    post_url = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/v2/sequence/search"
    query_sequence = {"sequence": sequence}

    try:
        response = requests.post(post_url, json=query_sequence)
        response.raise_for_status()  # Raise an exception if it fails
        job_id = response.json()["job_id"]
        print("Job ID:", job_id)
        if response.status_code == 200:
            print("Your search was successful")
            retrieve_results(job_id)
        elif response.status_code == 202:
            print("Your search is still in progress")
        else:
            print(f"Request failed with status code {response.status_code}")
            exit()
    except requests.RequestException as e:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Response text: {response.text}")
        exit()

def retrieve_results(job_id):
    get_url = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/v2/sequence/result"
    try:
        # Use requests.get for better error handling
        get_url = f"{get_url}?job_id={job_id}"
        print("Querying API with:", get_url)
        response = requests.get(get_url)
        response.raise_for_status()

        if response.status_code == 202:
            print("Your search is still in progress")
            return
        elif response.status_code == 200:
            print("Your search was successful")

        # Parse the JSON response with ijson
        items = ijson.items(response.content, 'item')
        for item in items:
            print(item)
    except requests.RequestException as e:
        print(f"An error occurred during the GET request: {e}")
        exit()


# Defining function for Ensembl search
def ensembl_search(ENSEMBL_ID):
    url = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/v2/ensembl/summary/" #url

    try:
        # Send GET request to retrieve Ensembl summary
        response = urlopen(f"{url}{ENSEMBL_ID}.json")
        if response.status == 200:
            print("Ensembl search successful")
        else:
            print(f"Ensembl search failed with status code {response.status}")

        r = ijson.parse(response)
        ensembls = ijson.items(r, "uniprot_mappings.item", use_float=True)

        # Loop for better printing
        for ensembl in ensembls:
            print(ensembl)
    except Exception as e:
        print(f"An error occurred during the Ensembl search: {e}")
        exit()

######## API
#url_API
WEBSITE_API = "https://www.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/api/v2/uniprot/summary/"

######## GOOGLE DRIVE
# Mounting the Google Drive to access files and directories
drive.mount('/content/drive')
destination_path = '/content/drive/MyDrive/3D_Beacons_files'

# Check whether the specified path exists or not
isExist = os.path.exists(destination_path)
if not isExist:
   # Create a new directory because it does not exist
   os.makedirs(destination_path)
   print("The new directory is created!")

def download_file(model_url, destination_file_path):
  try:
    urllib.request.urlretrieve(model_url, destination_file_path)
    print(f'File downloaded successfully to: {destination_file_path}')
  except urllib.error.URLError as e:
    print(f'Error downloading file: {e}')  # Handle URL-related errors
  except Exception as e:  # Catch other general exceptions
    print(f'An unexpected error occurred: {e}')




---



##1.&nbsp; BASIC SEARCH

This section implements API to search and filter predicted and experimentally determined structures available from the 3D-Beacons network.

Once you have found a model, you can download the specific

In [ ]:
#@title 1.1.&nbsp;  Get all macromolecular structures for a single entry
#@markdown  On this sub-section you will be able to retrieve all available structures and metainformation in 3D-Beacons from a single Uniprot accession ID.
#Run this code to display the widget
Uniprot_ID = "Q9I1F6" #@param {type:"string"}

#Run this code to search the network using the input from the widget
structures = Search3DBeacons(Uniprot_ID)

#print available structures from different providers
for structure in structures:
    print(structure)

### 1.2.&nbsp;  Performing one  filter

In [ ]:
#@title 1.2.1.  Filter by Model type
#@markdown  Results can be filtered according to the model category in the 3D-Beacons Network. Models are classified as:

#@markdown * EXPERIMENTALLY DETERMINED
#@markdown * CONFORMATIONAL ENSEMBLE
#@markdown * TEMPLATE-BASED
#@markdown * AB-INITIO

#@markdown Code below allows you to filter the previous search using a model category.
#@markdown
model_type = "EXPERIMENTALLY_DETERMINED" #@param ["CONFORMATIONAL_ENSEMBLE", "EXPERIMENTALLY_DETERMINED", "TEMPLATE-BASED", "AB-INITIO"]
model_type_modified= model_type.replace("_", " ")

#Run this code to filter according to model category
for structure in structures:
    model_category = structure.get('summary', {}).get('model_category')
    if model_category == model_type_modified:
        print(structure)

In [ ]:
#@title 1.2.2.&nbsp; Filter by Provider

#@markdown The 3D-Beacons Network facilitates the aggregation of coordinate files and metadata for both experimental and theoretical protein models. It encompasses a wide range of state-of-the-art and specialized model providers, as well as data from the European Protein Data Bank (PDBe).

#@markdown Model providers:
#@markdown * PDBe
#@markdown * SWISS-MODEL
#@markdown * AlphaFold DB
#@markdown * Genome3D
#@markdown * SASBDB
#@markdown * AlphaFill
#@markdown * ModelArchive
#@markdown * Protein Ensemble Database (PED)

#@markdown For further information, visit the [Documentation](https://wwwdev.ebi.ac.uk/pdbe/pdbe-kb/3dbeacons/docs) to check all the current providers.


provider = "AlphaFold_DB" #@param ["PDBe", "SWISS-MODEL", "AlphaFold_DB", "Genome3D", "SASBDB", "AlphaFill", "ModelArchive", "Protein_Ensemble_Database_(PED)"]
provider_modified= provider.replace("_", " ")

#Run this code to filter results according to given provider
for structure in structures:
    provider = structure.get('summary', {}).get('provider')

    if provider == provider_modified:
        print(structure)

### 1.3.&nbsp;  Performing two filters

This section uses two sets of filters to display the models available in the network

In [ ]:
#@title 1.3.1  Filter by Model type and Coverage

#markdown The code below will filter all the structures from a Uniprot ID and will save the model with the highest coverage to Google Drive.
import urllib.request

Uniprot_ID = "P04637"  # @param {type:"string"}
model_type = "CONFORMATIONAL_ENSEMBLE"  # @param ["CONFORMATIONAL_ENSEMBLE", "EXPERIMENTALLY_DETERMINED", "TEMPLATE-BASED", "AB-INITIO"]
model_type_modified = model_type.replace("_", " ")

#search and filter
response = urlopen(f"{WEBSITE_API}{Uniprot_ID}.json")
r = ijson.parse(response)
structures = list(ijson.items(r, "structures.item", use_float=True))
structures.sort(key=lambda x: x.get('summary', {}).get('coverage', 0), reverse=True)

highest_coverage_structure = None
for structure in structures:
    model_category = structure.get('summary', {}).get('model_category')
    if model_category == model_type_modified:
        highest_coverage_structure = structure
        break

# Check if a structure was found before accessing its properties
if highest_coverage_structure is not None:
    print( highest_coverage_structure)

else:
    print(f"No structure found with model type: {model_type_modified}")


In [ ]:
#@title 1.3.2&nbsp;  Filter by model type and retrieve results according to resolution.

#@markdown The code below will filter all the structures from a Uniprot ID and will save the models with a resolution higher than the specified on the widget.
# Run this code to display widgets
Uniprot_ID = "P04637"  # @param {type:"string"}
provider = "PDBe" #@param ["PDBe", "SWISS-MODEL", "AlphaFold_DB", "Genome3D", "SASBDB", "AlphaFill", "ModelArchive", "Protein_Ensemble_Database_(PED)"]
provider_modified= provider.replace("_", " ")
resolution_search = 1.3 # @param {type:"slider", min:0.1, max:5.0, step:0.1}

#Run this code to search and filter results according to the input provided
r = ijson.parse(urlopen(f"{WEBSITE_API}{Uniprot_ID}.json")) #build
structures = list(ijson.items(r, "structures.item", use_float=True))

#filter for experimental structures
high_resolution_structures = []
for structure in structures:
    provider = structure.get('summary', {}).get('provider')
    resolution = structure.get('summary', {}).get('resolution')

    if provider == provider_modified and resolution is not None and resolution < resolution_search:
        # Append the structure to the list without assigning the result back to the list
        high_resolution_structures.append(structure)

# Perform filter with the high-resolution structures
for structure in high_resolution_structures:
    print(structure)



---



##2.&nbsp; BASIC SEARCH

In [ ]:
#@title Perform sequence search

#@markdown The 3D-Beacons Network has introduced sequence similarity search functionality which allows you to query the network using the amino acid sequence of a protein.

#@markdown The Sequence Similarity Search option available through the network uses the Basic Local Alignment Search Tool (BLAST, Altschul et al., 1990) to find regions of sequence similarity by aligning them with a query sequence. By evaluating the match between the network and query sequence, valuable insights into the structure, function, and evolutionary aspects can be obtained, thus facilitating targeted and systematic exploration of protein structures.

#@markdown The code presented below allows you to search the network by performing a sequence-based search via API.

sequence = "MNMLVINGTPRKHGRTRIAASYIAALYHTA"# @param {type:"string"}

#Run this code to perform sequence search, it will use the AA from the widget
sequence_search(sequence)



---



##3.&nbsp; RETRIEVING ENSEMBL SUMMARY VIA 3D-BEACONS



In [ ]:
#@title Retrieve summary using Ensembl Identifier
#@markdown The 3D-Beacons network provides the resource for retrieving Ensembl summaries to explore the diverse transcript variants associated with a given gene. ENSEMBL provides information about the different transcript variants for a given gene that map to an UniProt ID.

Ensembl_ID_search= "ENSG00000288864" # @param {type:"string"}

ensembl_search(Ensembl_ID_search)



---



##4.&nbsp; DOWNLOAD SPECIFIC MODEL

The downloaded model can be found in the folder "3D_Beacons_files" in your Google Drive

In [ ]:
#@title Download
#@markdown Once you found the model/ models you need, you can customise this code to download the MMCIF file to Google Drive

#@markdown If multiple models are required, separated them with a comma.
Uniprot_ID = "P04637"# @param {type:"string"}
model_download = "3d06, 4mzi"  # @param {type:"string"}
model_download_modified = ", ".join(model_download.strip().split(","))
#@markdown The model_download can be found in the key 'model_identifier' within every summary

structures = Search3DBeacons(Uniprot_ID)
### Run this code to download the models, it will use the input from the widget
for structure in structures:
  model = structure.get('summary', {}).get('model_identifier')
  if model in model_download_modified:
    model_url = structure.get("summary", {}).get("model_url")
    model_format = structure.get("summary", {}).get("model_format")
    file_extension = '.cif' if model_format == 'MMCIF' else '.pdb'
    destination_file_path = os.path.join(destination_path, model + file_extension)
    download_file(model_url, destination_file_path)




---



# Using the AlphaFold API

**What are confidence scores?** AlphaFold and related tools compute several numbers that estimate how reliable the predicted structure is. This notebook computes five of them:
- **ipTM** – Overall confidence in chain positioning
- **ipSAE** – Stricter confidence using only well-predicted residues
- **pDockQ** – Structural plausibility (contact count + pLDDT)
- **pDockQ2** – Like pDockQ but also checks PAE at the interface
- **LIS** – Density of low-error inter-chain interactions

[Full API documentation](https://alphafold.ebi.ac.uk/api-docs)





In [ ]:
#@markdown Run this code block to initialise functions and variables used from this point onwards.
import requests

def display_api_results(results):
  for hit in results:
    print(hit)

def call_get_api(url_with_key):
  try:
    response = requests.get(url_with_key)
    response.raise_for_status()

    if response.status_code == 200:
      print("Your search was successful")
      return response.json()

    else:
      print("API called with", url_with_key)
      print(f"Request failed with status code {response.status_code}")
      return

  except requests.RequestException as e:
    print("API called with", url_with_key)
    print(f"Request failed with status code: {response.status_code}")
    print(f"Response text: {response.text}")

    return



##1.&nbsp; Fetch all AFDB models for a protein

Give API endpoint a UniProt accession (corresponds to a unique protein sequence in a given organism), and receive metadata and file download URLs.

In [ ]:
#@markdown Fetch AFDB model data for a protein

#@markdown Get a comprehensive set of metadata for a protein, along with URLs to download the structure.

def get_prediction_api(uniprot_accession):
  url = "https://alphafold.ebi.ac.uk/api/prediction/"
  url_completed = url + uniprot_accession

  return call_get_api(url_completed)

uniprot_accession = "P04637"# @param {type:"string"}

meta_data = get_prediction_api(uniprot_accession)
display_api_results(meta_data)


##2.&nbsp; Get data for based on an exact sequence match



In [ ]:
def get_sequence_match_api(sequence, q_type="sequence"):
  url = "https://alphafold.ebi.ac.uk/api/sequence/summary"
  url_completed = url + "?" + f"id={sequence}" + "&" f"?type={q_type}"

  return call_get_api(url_completed)


sequence = "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"# @param {type:"string"}

meta_data = get_sequence_match_api(sequence)
print(meta_data)



##3.&nbsp; Get data for complexes

In [ ]:
#@markdown Fetch AFDB model data for a protein complex

#@markdown Get a comprehensive set of metadata for a protein complex, along with URLs to download the structure. If monomers share the same UniProt accession, only complexes will be returned.

def get_complexes_api(uniprot_accession):
  url = "https://alphafold.ebi.ac.uk/api/complex/"
  url_completed = url + uniprot_accession

  return call_get_api(url_completed)

uniprot_accession = "A0A2G8KDW6"# @param {type:"string"}

meta_data = get_complexes_api(uniprot_accession)
display_api_results(meta_data)


##4.&nbsp; Downloading structures

In [ ]:
#@markdown Initialise functions for mmCIF, PAE, and pLDDT files from the AFDB:

def download_cif(meta):
  cif_url = meta['cifUrl']
  print(f'Downloading mmCIF: {cif_url}')
  cif_text = requests.get(cif_url, timeout=60).text
  return cif_text

def download_pae(meta):
  pae_url = meta['paeDocUrl']
  print(f'Downloading PAE:   {pae_url}')
  pae_raw = requests.get(pae_url, timeout=60).json()
  return pae_raw

def download_plddt(meta):
  plddt_url = meta['plddtDocUrl']
  print(f'Downloading pLDDT: {plddt_url}')
  plddt_raw = requests.get(plddt_url, timeout=60).json()
  return plddt_raw




In [ ]:
#@markdown Fetch list of models for a protein using the `prediction` API

uniprot_accession = "P04637"# @param {type:"string"}
meta_data = get_prediction_api(uniprot_accession)


In [ ]:
#@markdown Download mmCIFs, PAE matrices, and pLDDT scores for all hits
data = {}
for meta in meta_data:
  entity_id = meta["modelEntityId"]

  print(f"Downloading data for {entity_id}")
  cif = download_cif(meta)
  pae = download_pae(meta)
  plddt = download_plddt(meta)

  data[entity_id] = {
    "cif": cif,
    "pae": pae,
    "plddt": plddt
  }

In [ ]:
#@markdown Save results to file

for entry in data:
  entity_id = entry
  cif = data[entry]["cif"]

  with open(f'{entity_id}.cif', 'w') as f:
    f.write(cif)

  pae = data[entry]["pae"]

  with open(f'{entity_id}_pae.json', 'w') as f:
    json.dump(pae, f)

  plddt = data[entry]["plddt"]
  with open(f'{entity_id}_plddt.json', 'w') as f:
    json.dump(plddt, f)


In [ ]:
#@markdown Initialise functions for visualising protein structures

import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'molviewspec'],
    check=False
)
print('Package check complete.')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
import requests
import json
import warnings
import base64
from IPython.display import display, HTML, IFrame
import molviewspec as mvs

# ── Parse mmCIF (no BioPython required) ─────────────────────────────────────
def parse_mmcif_atoms(cif_text):
    """Return (records, col_idx) from the _atom_site loop in an mmCIF file."""
    lines = cif_text.split('\n')
    i = 0
    # Locate _atom_site loop
    while i < len(lines):
        if lines[i].strip() == 'loop_':
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines) and lines[j].strip().startswith('_atom_site.'):
                break
        i += 1
    else:
        raise ValueError('No _atom_site loop found in mmCIF')

    i += 1  # skip 'loop_' line
    columns = []
    while i < len(lines) and lines[i].strip().startswith('_atom_site.'):
        columns.append(lines[i].strip().split('_atom_site.')[1].strip())
        i += 1

    col_idx = {c: k for k, c in enumerate(columns)}
    n_cols = len(columns)

    records = []
    while i < len(lines):
        line = lines[i].strip()
        if not line or line.startswith('#'):
            i += 1
            continue
        if line.startswith('_') or line == 'loop_':
            break
        parts = line.split()
        if len(parts) >= n_cols:
            records.append(parts[:n_cols])
        i += 1
    return records, col_idx


def extract_ca_coords(records, col_idx):
    """Extract CA for coords, pLDDT, and residue info per chain."""
    ca_data = {}

    for rec in records:
        if rec[col_idx['group_PDB']] != 'ATOM':
            continue

        # Filter by model 1 if field exists
        if 'pdbx_PDB_model_num' in col_idx:
            if rec[col_idx['pdbx_PDB_model_num']] != '1':
                continue
        atom_name = rec[col_idx['label_atom_id']]
        if atom_name != "CA":
            continue
        chain   = rec[col_idx['label_asym_id']]
        res_str = rec[col_idx['label_seq_id']]
        if res_str in ('.', '?') or not res_str.lstrip('-').isdigit():
            continue
        resnum  = int(res_str)
        resname = rec[col_idx['label_comp_id']]
        coord   = np.array([float(rec[col_idx['Cartn_x']]),
                            float(rec[col_idx['Cartn_y']]),
                            float(rec[col_idx['Cartn_z']])])
        plddt   = float(rec[col_idx['B_iso_or_equiv']])
        entry   = {'coord': coord, 'resname': resname, 'plddt': plddt}
        key = (chain, resnum)
        if atom_name == 'CA':
            ca_data[key] = entry

    # Prefer CB; fall back to CA (covers GLY)
    combined = {k: ca_data.get(k)
                for k in set(list(ca_data))}

    chain_map = {}
    for (chain, resnum), data in combined.items():
        chain_map.setdefault(chain, []).append((resnum, data))

    ch_coords, ch_resids, ch_resnames, ch_plddt = {}, {}, {}, {}
    for chain, res_list in chain_map.items():
        res_list.sort(key=lambda x: x[0])
        ch_resids[chain]   = [r for r, _ in res_list]
        ch_coords[chain]   = np.array([d['coord'] for _, d in res_list])
        ch_resnames[chain] = [d['resname'] for _, d in res_list]
        ch_plddt[chain]    = np.array([d['plddt'] for _, d in res_list])

    return ch_coords, ch_resids, ch_resnames, ch_plddt


# records, col_idx = parse_mmcif_atoms(cif_text)
# ch_coords, ch_resids, ch_resnames, ch_plddt = extract_cb_coords(records, col_idx)


def show_mol_view(state, label, width=950, height=600):
    """Render a MolViewSpec State inline via a base64 data-URI iframe (works in PyCharm)."""
    html = state.molstar_html()
    encoded = base64.b64encode(html.encode()).decode()
    display(HTML(f'<div style="margin:10px 0 4px; font-weight:bold;">{label}</div>'))
    display(IFrame(src=f'data:text/html;base64,{encoded}', width=width, height=height))


# ── View 1: Chain overview ─────────────────────────────────────────────────
def show_default_structure(struct_url, struct_fmt):
  builder = mvs.create_builder()
  structure = (
      builder
      .download(url=struct_url)
      .parse(format=struct_fmt)
      .model_structure()
  )
  # Chain A — teal
  (structure
    .component(selector=mvs.ComponentExpression(label_asym_id='A'))
    .representation(type='cartoon')
    .color(color='#009688'))

  show_mol_view(builder.get_state(), 'View 1: Chain Overview (teal=A, coral=B, amber=interface)')


def parse_plddt(plddt_raw):
  # ── Parse pLDDT JSON ─────────────────────────────────────────────────────────
  plddt_scores   = np.array(plddt_raw['confidenceScore'], dtype=np.float32)
  plddt_residues = np.array(plddt_raw['residueNumber'], dtype=np.int32)
  plddt_chains_info = plddt_raw.get('chains', [])

  return plddt_scores, plddt_residues, plddt_chains_info

# ── View 2: pLDDT mapping ──────────────────────────────────────────────────
def show_structure_with_plddt(model_metadata, struct_fmt):
    builder2 = mvs.create_builder()

    struct_url = model_metadata['cifUrl']
    structure2 = (
        builder2
        .download(url=struct_url)
        .parse(format=struct_fmt)
        .model_structure()
    )
    plddt_colours = [
        (90, 100, '#1565C0'),   # dark blue
        (70,  90, '#42A5F5'),   # light blue
        (50,  70, '#FFCA28'),   # yellow
        ( 0,  50, '#EF6C00'),   # orange
    ]

    cif_text = download_cif(model_metadata)
    # pae = download_pae(meta)
    # plddt = download_plddt(meta)

    records, col_idx = parse_mmcif_atoms(cif_text)
    ch_coords, ch_resids, ch_resnames, ch_plddt = extract_ca_coords(records, col_idx)

    # resids_arr_ = ch_resids[cA_id if chain_label == 'A' else cB_id]
    plddt_arr = ch_plddt["A"]
    for lo, hi, colour in plddt_colours:
        idx_range = np.where((plddt_arr >= lo) & (plddt_arr < hi))[0]
        for i in idx_range:
            r = ch_resids["A"][i]
            (structure2
              .component(
                  selector=mvs.ComponentExpression(
                    label_asym_id="A",
                    beg_label_seq_id=r,
                    end_label_seq_id=r
                  )
                )
              .representation(type='cartoon')
              .color(color=colour))
    show_mol_view(builder2.get_state(), 'View 2: pLDDT Mapping (dark blue>90, light blue 70–90, yellow 50–70, orange<50)')


In [ ]:
#@markdown Render one isoform in 3D

show_default_structure(meta_data[0]['cifUrl'], 'mmcif')
show_structure_with_plddt(meta_data[0], 'mmcif')

## **Challenge**: Which isoform is the best structure to proceed with?

# TROUBLESHOOTING

| Problem / Response code| Possible cause | Solution |
| :---------: |  :- | :- |
| 202 | **Accepted request.** <br> 1. The sequence search submission was correct, and the job has been assigned a job identifier. <br> 2. The sequence search job is currently running.| 1.Please wait until the sequence search run completes. It can take 5-10+ minutes.
| 400 | **Bad request** <br>  1. Malformed UniProt accession.  <br>  2. Possibly invalid sequence.  <br>  3. Job identifier not found.|1. Please check that the input UniProt accession is correct. <br>  2. Please check your input sequence and retry the submission. <br> 3. Please check if your job identifier is correct.
| 404| **Not found.** <br> No results found for the given request.| 1. There may be no results for a specific UniProt accession or protein sequence
|500| **Internal Server Error** | 1. This error might be due to scheduled maintenance or rarely technical issues. <br> 2. Please try again later. If the issue persists, please email pdbekb_help@ebi.ac.uk

